# DP3 GainCal Python Binding

In [ ]:
import os
import shutil
import dp3
import numpy as np

from ska_sdp_func_python.calibration import (
    dp3_gaincal,
)

from ska_sdp_instrumental_calibration.data_managers.visibility import (
    load_ms_as_dataset_with_time_chunks,
)
from ska_sdp_instrumental_calibration.data_managers.gaintable import (
    create_gaintable_from_visibility,
)

from ska_sdp_datamodels.calibration import GainTable
from ska_sdp_datamodels.visibility import Visibility

In [ ]:
# Modify as required
ms_path = "/home/ska/Work/data/INST/lg3/cal_bpp_vis-lg3-rotated.small.ms"
msout_path = "dp3_output.ms"

skymodel_path = "./dp3_skymodel.txt"
parset_file = "./gaincal.parset"

func_python_output_gainpath = "gaincal_via_func_python.h5parm"
dp3_output_gainpath = "gaincal_via_execute.h5parm"

In [ ]:
if os.path.exists(msout_path):
    shutil.rmtree(msout_path)

if os.path.exists(msout_path):
    os.remove(parset_file)

In [ ]:
input_vis = load_ms_as_dataset_with_time_chunks(ms_path, -1).load()
init_gaintable = create_gaintable_from_visibility(
    input_vis, timeslice="full", jones_type="B", skip_default_chunk=True
).load()

## Calling dp3_gaincal from func-python

In [ ]:
print("==============================================")
print(" DP3 GainCal execution started.")
print("==============================================")

func_python_vis_corrected = dp3_gaincal(
    input_vis, "B", False, skymodel_path, func_python_output_gainpath
)

print("==============================================")
print("DP3 GainCal execution completed.")
print("==============================================")

## DP3 via in-memory python bindings 

In [ ]:
def dp3_gaincal_solve(
    self,
    vis_vis: np.ndarray,
    vis_flags: np.ndarray,
    vis_weight: np.ndarray,
    model_vis: np.ndarray | None,
    model_flags: np.ndarray | None,
    gain_gain: np.ndarray,
    gain_weight: np.ndarray,
    gain_residual: np.ndarray,
    ant1: np.ndarray,
    ant2: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Parameters
    ----------
    vis_vis : np.ndarray
        Observed visibility data. Shape: (time, baseline, freq, pol).
    vis_flags : np.ndarray
        Flags for observed visibilities. Shape matches `vis_vis`.
    vis_weight : np.ndarray
        Weights for observed visibilities. Shape matches `vis_vis`.
    model_vis : np.ndarray
        Model visibility data. Shape matches `vis_vis`.
    model_flags : np.ndarray
        Flags for model visibilities. Shape matches `vis_vis`.
    gain_gain : np.ndarray
        Initial gain estimates. Shape: (time, ant, freq, rec1, rec2).
    gain_weight : np.ndarray
        Storage for gain weights. Shape matches `gain_gain`.
    gain_residual : np.ndarray
        Storage for gain residuals. Shape matches `gain_gain`.
    ant1 : np.ndarray
        Indices of antenna 1 for each baseline. Shape: (nbl,).
    ant2 : np.ndarray
        Indices of antenna 2 for each baseline. Shape: (nbl,).

    Returns
    -------
    tuple of np.ndarray
        A tuple containing:

        - Updated gain array.
        - Gain weights.
        - Gain residuals.

    """
    pass

In [ ]:
from ska_sdp_instrumental_calibration.data_managers.visibility import (
    load_ms_as_dataset_with_time_chunks,
)

from ska_sdp_instrumental_calibration.data_managers.gaintable import (
    create_gaintable_from_visibility,
)

_gain, _weight, _residual = dp3_gaincal_solve(
    input_vis.vis.data,
    input_vis.flags.data,
    input_vis.weight.data,
    None,
    None,
    init_gaintable.gain.data,
    init_gaintable.weight.data,
    init_gaintable.residual.data,
    input_vis.antenna1.data,
    input_vis.antenna2.data,
)

In [ ]:
dp3_gaincal_gaintable = init_gaintable.copy(deep=True)

dp3_gaincal_gaintable.gain.data = _gain
dp3_gaincal_gaintable.weight.data = _weight
dp3_gaincal_gaintable.residual.data = _residual

## DP3 via execute

In [ ]:
print("==============================================")
print(" Running DP3 C++ GainCal from Python")
print("==============================================")

parset_content = f"""
msin = "{ms_path}"
msin.datacolumn = DATA
msin.chunksize = 500
msout = "{msout_path}"

steps = [cal]

cal.type = gaincal
cal.sourcedb = "{skymodel_path}"
cal.parmdb = "{dp3_output_gainpath}"
cal.caltype = diagonal
cal.solint = 0
cal.nchan = 1
"""

with open(parset_file, "w") as file:
    file.write(parset_content)

print(f"Parset: {parset_file}")

# Python -> pybind11 -> DP3 C++
dp3.execute(parset_file, [])

print()
print("DP3 GainCal execution completed.")

## Comparision

In [ ]:
from notebook_utils import create_gaintable_from_h5param

dp3_gaintable: GainTable = create_gaintable_from_h5param(
    dp3_output_gainpath, init_gaintable.soln_interval_slices, input_vis
)

In [ ]:
import xarray as xr

xr.testing.assert_allclose(dp3_gaincal_gaintable, dp3_gaintable)